Configurando o ambiente

In [15]:
# Exemplo do que fazer
import pandas as pd
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

#Diretórios
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

print("Ambiente configurado")
print("Arquivos do Kaggle na pasta data/raw")
orders = pd.read_csv("/workspaces/olist-bi-dashboard/data/raw/olist_orders_dataset.csv")


Ambiente configurado
Arquivos do Kaggle na pasta data/raw


Carregamento dos Dados Brutos - Dataset olist

In [25]:
#Carregar as tabelas 
orders = pd.read_csv("/workspaces/olist-bi-dashboard/data/raw/olist_orders_dataset.csv")
order_items = pd.read_csv("/workspaces/olist-bi-dashboard/data/raw/olist_order_items_dataset.csv")
order_reviews = pd.read_csv("/workspaces/olist-bi-dashboard/data/raw/olist_order_reviews_dataset.csv")
customers = pd.read_csv("/workspaces/olist-bi-dashboard/data/raw/olist_customers_dataset.csv")
category_translation = pd.read_csv("/workspaces/olist-bi-dashboard/data/raw/product_category_name_translation.csv")
sellers = pd.read_csv("/workspaces/olist-bi-dashboard/data/raw/olist_sellers_dataset.csv")

print("Tabelas carregadas:")
print(f"  orders:          {orders.shape}")
print(f"  order_items:     {order_items.shape}")
print(f"  order_reviews:   {order_reviews.shape}")
print(f"  customers:       {customers.shape}")
print(f"  sellers:         {sellers.shape}")


Tabelas carregadas:
  orders:          (99441, 8)
  order_items:     (112650, 7)
  order_reviews:   (99224, 7)
  customers:       (99441, 5)
  sellers:         (3095, 4)


In [27]:
# Visão geral rápida de cada tabela
for name, df in [("orders", orders), ("order_items", order_items), ("order_reviews", order_reviews)]:
    print(f"\n{'='*50}")
    print(f" {name}")
    print(f"  Linhas: {len(df):,}  |  Colunas: {df.shape[1]}")
    print(f"  Nulos: {df.isnull().sum().sum():,}")
    print(df.dtypes.to_string())



 orders
  Linhas: 99,441  |  Colunas: 8
  Nulos: 4,908
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str

 order_items
  Linhas: 112,650  |  Colunas: 7
  Nulos: 0
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64

 order_reviews
  Linhas: 99,224  |  Colunas: 7
  Nulos: 145,903
review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str


ETL - Limpeza dos dados

Tabela orders

In [17]:
# 4. Salvar limpo
orders.to_csv("/workspaces/olist-bi-dashboard/data/processed/orders_clean.csv", index=False)

In [18]:
import sqlite3
conn = sqlite3.connect("/workspaces/olist-bi-dashboard/data/olist.db")

In [19]:
# Lê a query do arquivo .sql
with open("/workspaces/olist-bi-dashboard/SQL/queries.sql") as f:
    queries = f.read()

# Executa uma query específica
receita = pd.read_sql("""
    SELECT
        strftime('%Y-%m', order_purchase_timestamp) AS mes,
        ROUND(SUM(price + freight_value), 2) AS receita_total
    FROM orders o
    JOIN order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY mes
    ORDER BY mes
""", conn)

receita.tail(10)

DatabaseError: Execution failed on sql '
    SELECT
        strftime('%Y-%m', order_purchase_timestamp) AS mes,
        ROUND(SUM(price + freight_value), 2) AS receita_total
    FROM orders o
    JOIN order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY mes
    ORDER BY mes
': no such table: orders